# Payer Policy Intelligence Pipeline
### End-to-End GenAI Pipeline — Hackathon Submission

**Models:** `llama-3.1-8b-instant` (bulk) · `llama-3.3-70b-versatile` (complex/retry) via Groq API  
**Output:** `outputs/result.csv` · `outputs/dashboard.html`

---
| Stage | Description |
|---|---|
| S1 | PDF Ingestion (pdfplumber + PyMuPDF fallback) |
| S2 | Brand Detection & Text Segmentation |
| S3 | Groq API Parameter Extraction (12 fields + cache) |
| S4 | Access Quality Score Computation (0–100) |
| S5 | Output Assembly, Validation & Dashboard |

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'groq>=1.4.0', 'pdfplumber>=0.10.0', 'pymupdf>=1.23.0',
    'pandas>=2.0.0', 'openpyxl>=3.1.0', 'python-dotenv>=1.0.0', 'tqdm>=4.66.0', '-q'])
print('All dependencies installed.')

## Cell 1 — Imports & Configuration

In [ ]:
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path

import pandas as pd
import pdfplumber
import fitz          # PyMuPDF
from tqdm.notebook import tqdm
from groq import Groq
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# ─── Paths ───────────────────────────────────────────────────
BASE_DIR       = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = BASE_DIR / 'Data'
OUTPUT_DIR     = BASE_DIR / 'outputs'
BUSINESS_RULES = BASE_DIR / 'PA_Business_rules_pc.xlsx'
OUTPUT_DIR.mkdir(exist_ok=True)

CACHE_FILE     = OUTPUT_DIR / 'extraction_log.json'
RESULT_CSV     = OUTPUT_DIR / 'result.csv'
SCORE_CSV      = OUTPUT_DIR / 'score_breakdown.csv'
ERROR_LOG      = OUTPUT_DIR / 'ingestion_errors.log'
DASHBOARD_HTML = OUTPUT_DIR / 'dashboard.html'

# ─── Models ──────────────────────────────────────────────────
MODEL_FAST   = 'llama-3.1-8b-instant'     # 14,400 RPD
MODEL_STRONG = 'llama-3.3-70b-versatile'  # 1,000 RPD

# ─── API Key (loads from .env automatically) ─────────────────
load_dotenv(BASE_DIR / '.env')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')
if not GROQ_API_KEY:
    raise EnvironmentError('GROQ_API_KEY not set. Add it to .env or run: $env:GROQ_API_KEY="gsk_..."')

client = Groq(api_key=GROQ_API_KEY)
print(f'Groq client ready. Data dir: {DATA_DIR}')
print(f'PDFs found: {len(list(DATA_DIR.glob("*.pdf")))}')

## Cell 2 — Load Pipeline Module

In [ ]:
# Import the full pipeline from pipeline.py
sys.path.insert(0, str(BASE_DIR))
import importlib, pipeline
importlib.reload(pipeline)   # ensures latest changes are loaded

from pipeline import (
    stage1_ingest_pdfs,
    stage2_build_jobs,
    stage3_extract_all,
    stage4_score_all,
    stage5_output,
    compute_access_score,
    BRAND_KEYWORDS,
    OUTPUT_COLUMNS,
)
print('Pipeline module loaded successfully.')

## Stage 1 — PDF Ingestion
Extracts full text from all PDFs using `pdfplumber` (primary) with `PyMuPDF` as fallback for scanned pages.

In [ ]:
raw_texts = stage1_ingest_pdfs()

# Quick QA
lengths = {fname: len(text) for fname, text in raw_texts.items()}
print(f'\nText length stats (chars):')
vals = list(lengths.values())
print(f'  Min: {min(vals):,}  |  Max: {max(vals):,}  |  Mean: {sum(vals)//len(vals):,}')

# Show a sample from the first PDF
first = list(raw_texts.keys())[0]
print(f'\nSample from {first}:')
print(raw_texts[first][:500])

## Stage 2 — Brand Detection & Job Segmentation
Loads the exact (Filename, Brand) job list from the Submissions sheet, then segments each PDF's text to the most brand-relevant portion.

In [ ]:
jobs = stage2_build_jobs(raw_texts)

# Preview first few jobs
print(f'\nFirst 5 jobs:')
for fname, brand, text in jobs[:5]:
    print(f'  {fname} | {brand} | {len(text):,} chars')

## Stage 3 — Groq API Parameter Extraction
Calls Groq API for each job with a structured JSON prompt. Results are cached to `extraction_log.json` — re-runs don't burn API quota.

- `llama-3.1-8b-instant` → text ≤ 8,000 chars (bulk, 14,400 RPD)
- `llama-3.3-70b-versatile` → text > 8,000 chars or JSON parse retry (1,000 RPD)

In [ ]:
extracted = stage3_extract_all(jobs)

# Show a sample extraction
sample_key = list(extracted.keys())[0]
print(f'\nSample extraction for {sample_key}:')
sample = {k: v for k, v in extracted[sample_key].items() if not k.startswith('_')}
for k, v in sample.items():
    print(f'  {k:40s}: {str(v)[:80]}')

## Stage 4 — Access Quality Score Computation
Applies the deterministic scoring rubric programmatically. Drug exclusions are set to 0.

In [ ]:
scores = stage4_score_all(extracted)

# Show score breakdown for first few jobs
print('\nScore breakdowns (first 5):')
print(f'{"File + Brand":<45} {"Score":>6}  Components')
print('-' * 100)
for (fname, brand), (score, bd) in list(scores.items())[:5]:
    label = f'{fname} | {brand}'
    components = '  '.join(f'{k}={v}' for k, v in bd.items())
    print(f'{label:<45} {score:>6}  {components}')

## Stage 5 — Output Assembly, Validation & Dashboard
Compiles all rows → validates → exports `result.csv` + `score_breakdown.csv` + `dashboard.html`.

In [ ]:
result_df = stage5_output(jobs, extracted, scores)
print('\nResult preview:')
result_df.head(10)

## Final Validation & Summary

In [ ]:
print('=== SUBMISSION CHECKLIST ===')
checks = [
    ('result.csv exists',           RESULT_CSV.exists()),
    ('score_breakdown.csv exists',  SCORE_CSV.exists()),
    ('extraction_log.json exists',  CACHE_FILE.exists()),
    ('dashboard.html exists',       DASHBOARD_HTML.exists()),
    ('All jobs have output rows',   len(result_df) >= len(jobs)),
    ('No duplicate (File, Brand)',  not result_df.duplicated(['Filename','Brand']).any()),
    ('No blank cells',              result_df.isnull().sum().sum() == 0),
    ('Scores in [0,100]',           ((result_df['Access Score']>=0)&(result_df['Access Score']<=100)).all()),
]
all_pass = True
for name, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'  [{status}] {name}')

print(f'\nOverall: {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')
print(f'\nResult CSV: {RESULT_CSV}')
print(f'Dashboard : {DASHBOARD_HTML}')

In [ ]:
# Score distribution
print('Score distribution:')
bins = [0,25,50,75,100]
labels = ['0-24 (Excluded/Highly Restrictive)','25-49 (Restrictive)','50-74 (Moderate)','75-100 (Permissive)']
result_df['score_band'] = pd.cut(result_df['Access Score'], bins=[-1]+bins, labels=labels, right=True)
print(result_df['score_band'].value_counts().to_string())

print('\nBrand averages:')
print(result_df.groupby('Brand')['Access Score'].mean().round(1).sort_values(ascending=False).to_string())